In [62]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")


In [35]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F18C91AAA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F18E7B7910>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [36]:
from langchain_core.messages import HumanMessage,SystemMessage
model.invoke([HumanMessage(content="Hi, My Name is Shreshth")])

AIMessage(content='Nice to meet you, Shreshth. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 43, 'total_tokens': 68, 'completion_time': 0.029989571, 'completion_tokens_details': None, 'prompt_time': 0.002595544, 'prompt_tokens_details': None, 'queue_time': 0.158479115, 'total_time': 0.032585115}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd3-7872-7b83-b84d-a13f6ef5caf2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 25, 'total_tokens': 68})

In [37]:
from langchain_core.messages import AIMessage
model.invoke(
    [
    HumanMessage(content="Hi, My Name is Shreshth"),
    AIMessage(content="Hello Shreshth! It's nice to meet you. \n\nAs a Chief AI Engineer."),
    HumanMessage(content="Hey What's my Name and what do I do?")
])

AIMessage(content="You told me earlier that your name is Shreshth. However, I couldn't find any information about your profession or what you do. Would you like to share that with me? I'm all ears!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 83, 'total_tokens': 126, 'completion_time': 0.063352028, 'completion_tokens_details': None, 'prompt_time': 0.004972642, 'prompt_tokens_details': None, 'queue_time': 0.159101916, 'total_time': 0.06832467}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd3-79b7-7522-84e2-8f6a766507cc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 43, 'total_tokens': 126})

## Message history:


In [38]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [39]:
config={"configurable":{"session_id":"chat1"}}

In [40]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My Name is Shreshth")],
    config=config
)

In [41]:
response.content

'Nice to meet you, Shreshth. Is there something I can help you with today?'

In [42]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)

AIMessage(content='Your name is Shreshth.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 77, 'total_tokens': 85, 'completion_time': 0.010920508, 'completion_tokens_details': None, 'prompt_time': 0.004629646, 'prompt_tokens_details': None, 'queue_time': 0.044736884, 'total_time': 0.015550154}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd3-7c09-7310-8c01-9c41b8b95c3b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 8, 'total_tokens': 85})

In [43]:
## Changing the session_id:
config1={"configurable":{"session_id":"chat"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)

In [44]:
response.content

"I don't have any information about your name. I'm a conversational AI, and our conversation just started, so I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you about it!"

In [47]:
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="My name is John")],
    config=config1
)

In [49]:
response2=with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)
response2.content

'Your name is John.'

## Prompt Template:

In [50]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Hey You are helpful assistant. Answer all the questions to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [51]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content='Nice to meet you, Krish. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 56, 'total_tokens': 71, 'completion_time': 0.018974894, 'completion_tokens_details': None, 'prompt_time': 0.977160427, 'prompt_tokens_details': None, 'queue_time': 0.233619638, 'total_time': 0.996135321}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfe5-f937-7610-ac0b-02c71b332ad1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 15, 'total_tokens': 71})

In [52]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="My name is John")],
    config=config
)

response

AIMessage(content='Nice to meet you, John. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 55, 'total_tokens': 70, 'completion_time': 0.018516214, 'completion_tokens_details': None, 'prompt_time': 0.002816103, 'prompt_tokens_details': None, 'queue_time': 0.048213387, 'total_time': 0.021332317}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfe7-e17a-7490-829b-41237d07932b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 15, 'total_tokens': 70})

In [53]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Hey You are helpful assistant. Answer all the questions to the nest of your ability{language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
response.content

'नमस्कार कृष! (Namaste Krish) मैं आपकी मदद करने के लिए तैयार हूँ। क्या आप कोई विशिष्ट प्रश्न पूछना चाहते हैं या कुछ जानना चाहते हैं?'

In [54]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="hi My name is krish")],"language":"Hindi"},
    config=config
)
response.content

'नमस्ते कृष (Namaste Krish)! स्वागत है (Swagat hai)। यह पूछने से पहले कि मैं आपकी मदद कैसे कर सकता हूं, आपको लगता है कि मैं आपके प्रश्नों का उत्तर देने में सक्षम हूं।'

In [55]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="hi My name is krish")],"language":"Spanish"},
    config=config
)
response.content

"Hi Krish! It seems like we had a hello just a minute ago. What's on your mind today? Want to discuss something or need help with a question?"

## Managing the Conversational history:

In [56]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

d:\CODES\Gen AI course\envi\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\CODES\Gen AI course\envi\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shres\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [57]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"Unfortunately, I'm a large language model, I don't have any information about your personal preferences. I'm starting from a blank slate with each conversation. If you'd like to share, I can chat with you about your favorite ice cream flavor!"

In [58]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'The math problem you asked was 2 + 2.'

In [59]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [60]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't mention your name. I'm here to help, but I don't have any information about a specific user."

In [61]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem yet. I'm here to help with any questions or problems you might have, including math problems if you'd like to ask one."